In [ ]:
import glob
import os
import shutil

REPO = "/kaggle/working/small-model-alignment"
found = glob.glob("/kaggle/input/**/pa2-code/**/alignment_core.py", recursive=True)
print(found)
if not os.path.exists(REPO):
    shutil.copytree(os.path.dirname(found[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
print(os.listdir(REPO))

In [ ]:
!nvidia-smi
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
assert _exit_code == 0

In [ ]:
import datasets
import torch
import transformers

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())
print("model proxy available", "MODEL_PROXY_URL" in os.environ)

In [ ]:
!PYTHONPATH=. python -m pytest -q tests
assert _exit_code == 0

In [ ]:
!python scripts/prepare_eval_sets.py
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_base_alpaca.json --generator qwen2.5-0.5b-base --dataset alpaca_eval --limit 3 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/base_alpaca.json --generator qwen2.5-0.5b-base --dataset alpaca_eval --limit 4 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft --train_size 64 --val_size 16 --eval_every 1
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/sft/metrics.jsonl --output_dir outputs/plots --run_name sft --kind sft
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_sft_alpaca.json --generator qwen2.5-0.5b-sft --dataset alpaca_eval --limit 3 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sft_alpaca.json --generator qwen2.5-0.5b-sft --dataset alpaca_eval --limit 4 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sft_hh.json --generator qwen2.5-0.5b-sft --dataset hh_rlhf_style_eval --limit 4 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/sft_run_outputs.zip outputs eval_sets -x 'outputs/*/best/*' 'outputs/*/final/*'
assert _exit_code == 0

In [ ]:
!python scripts/train_dpo.py --dataset hh --sft_path outputs/sft/best --output_dir outputs/dpo_hh --train_size 64 --val_size 16 --eval_every 1
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/dpo_hh/metrics.jsonl --output_dir outputs/plots --run_name dpo_hh --kind dpo
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_hh/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sanity_dpo_hh_hh.json --generator qwen2.5-0.5b-dpo-hh --dataset hh_rlhf_style_eval --limit 3 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_hh/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/dpo_hh_hh.json --generator qwen2.5-0.5b-dpo-hh --dataset hh_rlhf_style_eval --limit 4 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/build_pairs.py --baseline_path outputs/generations/sft_hh.json --candidate_path outputs/generations/dpo_hh_hh.json --output_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/dpo_hh_run_outputs.zip outputs -x 'outputs/*/best/*' 'outputs/*/final/*'
assert _exit_code == 0

In [ ]:
!python scripts/train_dpo.py --dataset ultrafeedback --sft_path outputs/sft/best --output_dir outputs/dpo_ultrafeedback --train_size 64 --val_size 16 --eval_every 1
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/dpo_ultrafeedback/metrics.jsonl --output_dir outputs/plots --run_name dpo_ultrafeedback --kind dpo
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_ultrafeedback/best --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_dpo_uf_alpaca.json --generator qwen2.5-0.5b-dpo-ultrafeedback --dataset alpaca_eval --limit 3 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_ultrafeedback/best --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/dpo_uf_alpaca.json --generator qwen2.5-0.5b-dpo-ultrafeedback --dataset alpaca_eval --limit 4 --max_new_tokens 48 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/build_pairs.py --baseline_path outputs/generations/sft_alpaca.json --candidate_path outputs/generations/dpo_uf_alpaca.json --output_path outputs/eval/dpo_ultrafeedback_vs_sft_pairs.jsonl
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/dpo_ultrafeedback_run_outputs.zip outputs -x 'outputs/*/best/*' 'outputs/*/final/*'
assert _exit_code == 0

In [ ]:
!python -m pip install -q "trl==0.29.1" "peft>=0.17,<0.22" "transformers>=4.56.2,<5" "datasets>=3.0,<5" "accelerate>=1.4.0"
assert _exit_code == 0

In [ ]:
!python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo --epochs 1
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/lora_dpo/metrics.jsonl --output_dir outputs/plots --run_name lora_dpo --kind lora
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_dpo/final --output_path outputs/generations/sanity_smol_lora_persona.json --generator smollm2-360m-lora-dpo --limit 3 --eval_path eval_sets/persona_style_eval.json --dataset persona_style_eval --prompt_format chat --max_new_tokens 48 --repetition_penalty 1.1 --limit 4
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --output_path outputs/generations/smol_base_persona.json --generator smollm2-360m-instruct --eval_path eval_sets/persona_style_eval.json --dataset persona_style_eval --prompt_format chat --max_new_tokens 48 --repetition_penalty 1.1 --limit 4
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_dpo/final --output_path outputs/generations/smol_lora_persona.json --generator smollm2-360m-lora-dpo --eval_path eval_sets/persona_style_eval.json --dataset persona_style_eval --prompt_format chat --max_new_tokens 48 --repetition_penalty 1.1 --limit 4
assert _exit_code == 0

In [ ]:
!python scripts/build_pairs.py --baseline_path outputs/generations/smol_base_persona.json --candidate_path outputs/generations/smol_lora_persona.json --output_path outputs/eval/lora_dpo_vs_base_pairs.jsonl
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/lora_dpo_run_outputs.zip outputs -x 'outputs/sft/*' 'outputs/dpo_hh/*' 'outputs/dpo_ultrafeedback/*'
assert _exit_code == 0

In [ ]:
!rm -rf outputs/sft/best outputs/sft/final outputs/dpo_hh/best outputs/dpo_hh/final outputs/dpo_ultrafeedback/best outputs/dpo_ultrafeedback/final
assert _exit_code == 0